# Claims Analysis
Quick look at the claims data. Run cells top to bottom.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

## Step 1

In [ ]:
# Load data and do a quick first look
DF = pd.read_csv("C:/Users/analyst/Desktop/data/claims_sample.csv")

DF.head()
print(DF.shape)
print(DF.dtypes)
plt.hist(DF['claim_amount'].dropna(), bins=50)
plt.title('claim amount distribution')
plt.show()

## Data prep

In [ ]:
# must run after cell above

# clean up the BE region subset
BE_regions = ["BE-BRU", "BE-VLG"]
ClaimData = DF[DF['region'].isin(BE_regions)].copy()
ClaimData = ClaimData[ClaimData['claim_amount'].notna()]
ClaimData = ClaimData[~ClaimData['peril'].isin(['UNKNOWN', ''])]
ClaimData['claim_date'] = pd.to_datetime(ClaimData['claim_date'], format='mixed', dayfirst=True)
ClaimData['claim_amount'] = ClaimData['claim_amount'].astype(float)
ClaimData['premium'] = ClaimData['premium'].astype(float)
ClaimData = ClaimData.drop_duplicates(subset=['policy_id', 'claim_date'])
ClaimData = ClaimData.reset_index(drop=True)

print('BE subset shape:', ClaimData.shape)
ClaimData.head()

In [ ]:
# same cleaning for NL
NL_regions = ["NL-NH", "NL-ZH"]
Data = DF[DF['region'].isin(NL_regions)].copy()
Data = Data[Data['claim_amount'].notna()]
Data = Data[~Data['peril'].isin(['UNKNOWN', ''])]
Data['claim_date'] = pd.to_datetime(Data['claim_date'], format='mixed', dayfirst=True)
Data['claim_amount'] = Data['claim_amount'].astype(float)
Data['premium'] = Data['premium'].astype(float)
Data = Data.drop_duplicates(subset=['policy_id'])
Data = Data.reset_index(drop=True)

print('NL subset shape:', Data.shape)
Data.head()

In [ ]:
# same for DE
DE_Regions = ["DE-BAY", "DE-NRW"]
d = DF[DF['region'].isin(DE_Regions)].copy()
d = d[d['claim_amount'].notna()]
d = d[~d['peril'].isin(['UNKNOWN', ''])]
d['claim_date'] = pd.to_datetime(d['claim_date'], format='mixed', dayfirst=True)
d['claim_amount'] = d['claim_amount'].astype(float)
d['premium'] = d['premium'].astype(float)
d = d.drop_duplicates(subset=['policy_id', 'claim_date'])
d = d.reset_index(drop=True)

print('DE subset shape:', d.shape)
d.head()

## Analysis

In [ ]:
def RunAnalysis(filepath):
    DF2 = pd.read_csv(filepath)
    DF2 = DF2[DF2['claim_amount'].notna()]
    DF2 = DF2[~DF2['peril'].isin(['UNKNOWN', ''])]
    DF2['claim_date'] = pd.to_datetime(DF2['claim_date'], format='mixed', dayfirst=True)
    DF2['claim_amount'] = DF2['claim_amount'].astype(float)
    DF2['premium'] = DF2['premium'].astype(float)
    DF2 = DF2.drop_duplicates(subset=['policy_id', 'claim_date'])
    DF2 = DF2.reset_index(drop=True)
    DF2['LR'] = DF2['claim_amount'] / DF2['premium']
    allRegions = DF2['region'].unique()
    results = []
    for r in allRegions:
        x = DF2[DF2['region'] == r]
        avgLR = x['LR'].mean()
        totalAmt = x['claim_amount'].sum()
        results.append({'region': r, 'avg_lr': avgLR, 'total': totalAmt})
    Summary = pd.DataFrame(results)
    Summary = Summary.sort_values('avg_lr', ascending=False)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(Summary['region'], Summary['avg_lr'], color='steelblue')
    ax.set_xlabel('Region')
    ax.set_ylabel('Avg Loss Ratio')
    ax.set_title('Average Loss Ratio by Region')
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()
    highrisk = DF2[DF2['claim_amount'] > 15000]
    print(f'High-risk policies: {len(highrisk)}')
    print(highrisk[['policy_id', 'claim_amount', 'region', 'peril']].head(20))
    return Summary


Result = RunAnalysis("C:/Users/analyst/Desktop/data/claims_sample.csv")

## More analysis

In [ ]:
# flag high risk — threshold picked based on gut feel from earlier histogram
Flagged = DF[DF['claim_amount'] > 15000].copy()

# Flagged = DF[DF['claim_amount'] > 10000].copy()
# Flagged = DF[DF['claim_amount'] > 20000].copy()

Flagged['LR'] = Flagged['claim_amount'] / Flagged['premium']
print(Flagged[['policy_id', 'region', 'peril', 'claim_amount', 'LR']].sort_values('LR', ascending=False).head(30))

In [ ]:
# breakdown by peril for high-risk
# tried groupby first but had NaN issues, gave up and just did value_counts

# x = Flagged.groupby('peril')['claim_amount'].mean()
# print(x)

print(Flagged['peril'].value_counts())

# also tried this but got confused by the index
# for p in Flagged['peril'].unique():
#     sub = Flagged[Flagged['peril'] == p]
#     print(p, sub['claim_amount'].mean())

## Plot

In [ ]:
# loss ratio over time — split by year
DF['claim_date2'] = pd.to_datetime(DF['claim_date'], format='mixed', dayfirst=True)
DF['yr'] = DF['claim_date2'].dt.year
DF['LR2'] = DF['claim_amount'] / DF['premium']

YearSummary = []
for y in sorted(DF['yr'].dropna().unique()):
    sub = DF[DF['yr'] == y]
    YearSummary.append({'year': y, 'avg_lr': sub['LR2'].mean()})
YearSummary = pd.DataFrame(YearSummary)

plt.figure(figsize=(7, 4))
plt.plot(YearSummary['year'], YearSummary['avg_lr'], marker='o')
plt.title('Avg Loss Ratio per Year')
plt.xlabel('Year')
plt.ylabel('Avg LR')
plt.tight_layout()
plt.show()

# also show region table again for reference
print(Result)